# 03 - Classical Baselines

Train CPU-friendly classical models to establish non-neural baseline results.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'rareguard').exists()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate
    if Path('/content').exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception as exc:
            print('Google Drive mount skipped/failed:', exc)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. '
        'If you are using Colab connector, Colab cannot read /Volumes/LocalDrive1 directly. '
        'Copy/sync the IMPLEMENTATION folder to Google Drive at '
        '/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION, '
        'or run this notebook with a local VS Code Python kernel on your Mac.'
    )
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python path ready')
for name in ['logs', 'results', 'processed', 'features', 'models', 'figures', 'reports']:
    (PROJECT_ROOT / 'artifacts' / name).mkdir(parents=True, exist_ok=True)
print('Artifact folders ready:', PROJECT_ROOT / 'artifacts')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Python path ready
Artifact folders ready: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts


In [2]:
DATASET_KEY = 'all'
!python -m rareguard.experiments.run_baselines --config configs/default.yaml --datasets configs/datasets.yaml --dataset {DATASET_KEY}


2026-09-25 04:15:46,920 | INFO | rareguard | Finished baselines for cse_cic_ids2018
2026-09-25 04:16:03,263 | INFO | rareguard | Finished baselines for cic_bccc_nrc_tabulariotattack_2024
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
2026-09-25 04:19:23,161 | INFO | rareguard | Finished baselines for cic_iiot_2025_datasense
2026-09-25 04:19:48,751 | INFO | rareguard | Finished baselines for edge_iiotset
2026-09-25 04:19:48,779 | INFO | rareguard | Wrote artifacts/results/baseline_metrics.csv


In [3]:
import pandas as pd
metrics = pd.read_csv('artifacts/results/baseline_metrics.csv')
display(metrics.sort_values(['dataset', 'f1_macro'], ascending=[True, False]))


,dataset,model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
3,cic_bccc_nrc_tabulariotattack_2024,LogReg,1.000000,1.000000,1.000000,1.000000,1.000000
5,cic_bccc_nrc_tabulariotattack_2024,MLP,0.999868,0.999928,0.999250,0.999589,0.999868
4,cic_bccc_nrc_tabulariotattack_2024,RandomForest,0.999605,0.999784,0.997751,0.998765,0.999605
7,cic_iiot_2025_datasense,RandomForest,0.981000,0.939769,0.926394,0.931630,0.980829
8,cic_iiot_2025_datasense,MLP,0.970000,0.932365,0.858476,0.886788,0.969382
6,cic_iiot_2025_datasense,LogReg,0.949500,0.855895,0.898342,0.869349,0.950227
2,cse_cic_ids2018,MLP,0.993600,0.968767,0.897678,0.925296,0.993087
0,cse_cic_ids2018,LogReg,0.943000,0.858763,0.943827,0.871608,0.960475
1,cse_cic_ids2018,RandomForest,0.944300,0.860133,0.942633,0.871271,0.961280
9,edge_iiotset,LogReg,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
# Optional: sync artifacts to the configured Google Drive folder.
# Keep --dry-run first if you want to preview.
!python scripts/sync_to_drive.py --dry-run
# !python scripts/sync_to_drive.py


rclone is not installed in this runtime.
Artifacts are already being written inside Google Drive because the project root is under /content/drive.
Current artifact folder: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION/artifacts
Skipping external rclone mirror. Continue with the next notebook.
